# 📊 EDA & Data Analysis — Government Scheme Recommendation System

This notebook consolidates all **Exploratory Data Analysis (EDA)**, **Data Preprocessing**, **Feature Engineering**, and **Model Comparison** into a single interactive document.

---

### Sections:
1. [Dataset Overview](#1-dataset-overview)
2. [Univariate Analysis](#2-univariate-analysis)
3. [Bivariate Analysis](#3-bivariate-analysis)
4. [Data Preprocessing Pipeline](#4-data-preprocessing-pipeline)
5. [Feature Engineering](#5-feature-engineering)
6. [Feature Importance](#6-feature-importance)
7. [Model Comparison — Why Random Forest?](#7-model-comparison)


In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams.update({'font.size': 12, 'figure.dpi': 110})

print('Libraries loaded successfully!')

## 1. Dataset Overview

In [ ]:
import os
BASE_DIR = os.path.dirname(os.getcwd())  # Go up one level from notebooks/
USERS_CSV   = os.path.join(BASE_DIR, '..', 'scheme-recommender', 'users.csv')
SCHEMES_CSV = os.path.join(BASE_DIR, '..', 'datasets', 'schemes.csv')

df = pd.read_csv(USERS_CSV)
schemes_df = pd.read_csv(SCHEMES_CSV)

print(f'Users dataset : {df.shape[0]} rows × {df.shape[1]} columns')
print(f'Schemes dataset: {schemes_df.shape[0]} rows × {schemes_df.shape[1]} columns')
print(f'Target classes : {df["Recommended_Scheme"].nunique()} unique schemes\n')
df.head()

In [ ]:
# Data types & missing values
info_df = pd.DataFrame({
    'dtype': df.dtypes,
    'missing': df.isnull().sum(),
    'unique': df.nunique(),
    'sample': [df[c].iloc[0] for c in df.columns]
})
print(info_df.to_string())

In [ ]:
# Descriptive statistics
df.describe().round(2)

## 2. Univariate Analysis

In [ ]:
# Age & Income distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Numeric Features — Distribution Analysis', fontsize=15, fontweight='bold')

axes[0,0].hist(df['Age'], bins=20, color='#0ea5e9', edgecolor='white')
axes[0,0].set_title('Age Histogram'); axes[0,0].set_xlabel('Age')

axes[0,1].boxplot(df['Age'], patch_artist=True,
                  boxprops=dict(facecolor='#0ea5e9'),
                  medianprops=dict(color='white', linewidth=2))
axes[0,1].set_title('Age Box Plot')

axes[1,0].hist(df['Income']/1000, bins=25, color='#10b981', edgecolor='white')
axes[1,0].set_title('Income Histogram'); axes[1,0].set_xlabel('Income (K INR)')

axes[1,1].boxplot(df['Income']/1000, patch_artist=True,
                  boxprops=dict(facecolor='#10b981'),
                  medianprops=dict(color='white', linewidth=2))
axes[1,1].set_title('Income Box Plot')

plt.tight_layout()
plt.show()

In [ ]:
# Target variable — Scheme distribution
fig, ax = plt.subplots(figsize=(13, 6))
scheme_cnt = df['Recommended_Scheme'].value_counts()
colors = plt.cm.tab20.colors[:len(scheme_cnt)]
bars = ax.barh(scheme_cnt.index, scheme_cnt.values, color=colors)
ax.set_xlabel('Number of Citizens')
ax.set_title('Target Variable — Recommended Scheme Distribution', fontsize=14, fontweight='bold')
for bar, val in zip(bars, scheme_cnt.values):
    ax.text(val+2, bar.get_y()+bar.get_height()/2, str(val), va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Categorical columns grid
cats = ['Gender', 'Occupation', 'Category', 'Education', 'State', 'Rural_Urban']
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Categorical Feature Distributions', fontsize=15, fontweight='bold')
for ax, col in zip(axes.flatten(), cats):
    cnt = df[col].value_counts()
    ax.bar(cnt.index, cnt.values, color='#6366f1', edgecolor='white')
    ax.set_title(col, fontweight='bold')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## 3. Bivariate Analysis

In [ ]:
# Occupation vs Scheme — Heatmap
ct = pd.crosstab(df['Occupation'], df['Recommended_Scheme'])
fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(ct, annot=True, fmt='d', cmap='Blues', linewidths=0.5, ax=ax)
ax.set_title('Occupation × Recommended Scheme', fontsize=14, fontweight='bold')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Income distribution per scheme
fig, ax = plt.subplots(figsize=(14, 7))
scheme_order = df.groupby('Recommended_Scheme')['Income'].median().sort_values().index
groups = [df[df['Recommended_Scheme'] == s]['Income'].values / 1000 for s in scheme_order]
ax.boxplot(groups, labels=scheme_order, vert=False, patch_artist=True,
           boxprops=dict(facecolor='#e0f2fe', color='#0369a1'),
           medianprops=dict(color='#0369a1', linewidth=2))
ax.set_xlabel('Income (thousands INR)')
ax.set_title('Income Distribution by Recommended Scheme', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Age vs Income scatter
fig, ax = plt.subplots(figsize=(10, 6))
for occ, col in zip(df['Occupation'].unique(), ['#0ea5e9', '#10b981', '#f59e0b', '#6366f1']):
    sub = df[df['Occupation'] == occ]
    ax.scatter(sub['Age'], sub['Income']/1000, alpha=0.4, label=occ, color=col, s=45)
ax.set_xlabel('Age'); ax.set_ylabel('Income (K INR)')
ax.set_title('Age vs Income — by Occupation', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Data Preprocessing Pipeline

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# Step 1: Clean
df_clean = df.dropna().drop_duplicates().reset_index(drop=True)
print(f'After cleaning: {df_clean.shape[0]} rows')

# Step 2: Outlier capping (IQR)
df_proc = df_clean.copy()
for col in ['Age', 'Income']:
    Q1, Q3 = df_proc[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    df_proc[col] = df_proc[col].clip(Q1 - 1.5*IQR, Q3 + 1.5*IQR)
print('Outlier capping done (IQR Winsorization)')

# Step 3: Feature engineering
df_proc['Is_Low_Income'] = (df_proc['Income'] < 300000).astype(int)
df_proc['Is_Youth']      = (df_proc['Age'] <= 25).astype(int)
df_proc['Is_Vulnerable'] = ((df_proc['Rural_Urban'] == 'Rural') & (df_proc['Is_Low_Income'] == 1)).astype(int)
print('Feature engineering done')

# Step 4: Encode
CATEGORICAL_COLS = ['Gender', 'State', 'Education', 'Occupation',
                    'Category', 'Disability_Status', 'Rural_Urban', 'Employment_Status']
encoders = {}
for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    df_proc[col + '_enc'] = le.fit_transform(df_proc[col].astype(str))
    encoders[col] = le

le_y = LabelEncoder()
y_enc = le_y.fit_transform(df_proc['Recommended_Scheme'])

FEATURE_COLS = [
    'Age', 'Income',
    'Gender_enc', 'State_enc', 'Education_enc', 'Occupation_enc',
    'Category_enc', 'Disability_Status_enc', 'Rural_Urban_enc', 'Employment_Status_enc',
    'Is_Low_Income', 'Is_Youth', 'Is_Vulnerable'
]
X = df_proc[FEATURE_COLS]
y = y_enc

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'\nTrain: {X_train.shape[0]} | Test: {X_test.shape[0]}')
print(f'Features: {len(FEATURE_COLS)} | Classes: {len(le_y.classes_)}')

## 5. Feature Engineering

In [ ]:
# Show derived features
df_proc[['Age', 'Income', 'Rural_Urban', 'Is_Low_Income', 'Is_Youth', 'Is_Vulnerable']].head(10)

In [ ]:
# Income groups
df_proc['Income_Group'] = pd.qcut(df_proc['Income'], q=5,
    labels=['Very Low', 'Low', 'Middle', 'High', 'Very High'])
fig, ax = plt.subplots(figsize=(10, 5))
cnt = df_proc['Income_Group'].value_counts().sort_index()
ax.bar(cnt.index.astype(str), cnt.values, color='#6366f1', edgecolor='white')
ax.set_title('Income Group Distribution (Quintiles)', fontsize=14, fontweight='bold')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

## 6. Feature Importance

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

fi_df = pd.DataFrame({
    'Feature': FEATURE_COLS,
    'Importance (%)': (rf.feature_importances_ * 100).round(2)
}).sort_values('Importance (%)', ascending=True)

fig, ax = plt.subplots(figsize=(12, 7))
colors = ['#0ea5e9' if v > 15 else '#6366f1' if v > 5 else '#94a3b8'
          for v in fi_df['Importance (%)']]
bars = ax.barh(fi_df['Feature'], fi_df['Importance (%)'], color=colors)
ax.set_xlabel('Feature Importance (%)')
ax.set_title('Random Forest Feature Importance (Gini Impurity)', fontsize=14, fontweight='bold')
for bar, val in zip(bars, fi_df['Importance (%)']):
    ax.text(val + 0.2, bar.get_y() + bar.get_height()/2, f'{val:.2f}%', va='center', fontsize=9)
plt.tight_layout()
plt.show()

print(fi_df.sort_values('Importance (%)', ascending=False).to_string(index=False))

## 7. Model Comparison

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.preprocessing import StandardScaler
import time

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

MODELS = {
    'Random Forest': (RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1), False),
    'Decision Tree': (DecisionTreeClassifier(max_depth=15, random_state=42), False),
    'KNN (K=7)':     (KNeighborsClassifier(n_neighbors=7, weights='distance'), True),
    'SVM (RBF)':     (SVC(kernel='rbf', random_state=42), True),
    'Logistic Reg.': (LogisticRegression(max_iter=2000, random_state=42), True),
    'Naive Bayes':   (GaussianNB(), False),
}

results = []
for name, (mdl, scaled) in MODELS.items():
    Xtr = X_train_sc if scaled else X_train.values
    Xte = X_test_sc  if scaled else X_test.values
    t0 = time.time()
    mdl.fit(Xtr, y_train)
    elapsed = time.time() - t0
    preds = mdl.predict(Xte)
    results.append({
        'Model': name,
        'Accuracy %': round(accuracy_score(y_test, preds)*100, 2),
        'F1 %':       round(f1_score(y_test, preds, average='weighted', zero_division=0)*100, 2),
        'Precision %': round(precision_score(y_test, preds, average='weighted', zero_division=0)*100, 2),
        'Recall %':   round(recall_score(y_test, preds, average='weighted', zero_division=0)*100, 2),
        'Time (s)':   round(elapsed, 3)
    })

comp_df = pd.DataFrame(results).sort_values('Accuracy %', ascending=False).reset_index(drop=True)
print(comp_df.to_string(index=False))

In [ ]:
# Grouped bar chart
metrics = ['Accuracy %', 'F1 %', 'Precision %', 'Recall %']
x = np.arange(len(comp_df))
width = 0.2
colors_bar = ['#0ea5e9', '#10b981', '#f59e0b', '#6366f1']

fig, ax = plt.subplots(figsize=(15, 7))
for i, (metric, col) in enumerate(zip(metrics, colors_bar)):
    ax.bar(x + i*width, comp_df[metric], width, label=metric, color=col, alpha=0.85)

ax.set_xticks(x + width*1.5)
ax.set_xticklabels(comp_df['Model'], rotation=15, ha='right')
ax.set_ylim(0, 110)
ax.set_ylabel('Score (%)')
ax.set_title('Model Performance Comparison — Accuracy, F1, Precision, Recall',
             fontsize=14, fontweight='bold')
ax.legend()
ax.axhline(90, color='red', linewidth=1, linestyle='--', alpha=0.5, label='90% line')
plt.tight_layout()
plt.show()

In [ ]:
print('\n✅ Why Random Forest is selected:')
reasons = [
    'Highest Accuracy & F1-Score across all models',
    'Ensemble bagging prevents overfitting (100 decorrelated trees)',
    'Natively handles non-linear composite eligibility rules',
    'Gini feature importance enables explainable recommendations',
    'No scaling required — handles mixed categorical + numeric data',
    'Robust to outliers and missing inputs (averaging effect)',
    'Out-of-bag error provides free internal validation'
]
for i, r in enumerate(reasons, 1):
    print(f'  {i}. {r}')